# Autoencoders x PCA - Redução de Dimensionalidade

## 1. Introdução

Como vimos na parte explicativa (arquivo `Autoencoders.pdf`) sobre os autoencoders, eles podem ser utilizados para redução de dimensionalidade [2,3]. Esse notebook visa comparar como modelos de aprendizado de máquina (no caso, Floresta Aleatória, Support Vector Regression e Regressão Linear) desempenham (utilizamos como métrica RMSE) nos casos de redução de dimensionalidade por PCA e por Autoencoders. Para isso, utilizamos um dataset sintético do `scikit-learn`. 

O arquivo `Autoencoders_PCA_estudo.py` é uma generalização desse notebook, que varia o número de features informativas, comparando o desempenho dos modelos em PCA e Autoencoders. Ou seja, esse notebook é uma ilustração didática do estudo mais profundo que foi feito nesse arquivo, e está discutido em `Autoencoders.pdf`. 

A principal motivação para usar autoencoders para esse fim é que eles não são lineares, ao contrário do PCA. Isso pode fazer com que o autoencoder capture relações que o PCA não captura, e melhorar o desempenho dos modelos [2]. 

### 1.1 Mas por quê aplicar redução de dimensionalidade?

Datasets com dimensionalidade muito alta (ou seja, um número de features grande) trazem problemas como aumento do custo computacional, necessidade de muita memória pra armazenar os dados e geralmente diminuem a capacidade de generalização dos modelos, reduzindo sua capacidade de desempenho. [5]

## 2. Importações e configurações

In [23]:
import numpy as np
from sklearn.datasets import make_regression
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt

In [24]:
SEMENTE = 3737

torch.manual_seed(SEMENTE)
np.random.seed(SEMENTE)

## 3. Gerando dados sintéticos

In [25]:
X, y = make_regression(
    n_samples=5000,
    n_features=100,
    n_informative=40,
    noise=5,
    random_state=SEMENTE
)

Temos então 5000 amostras, 100 features (das quais apenas 40 são verdadeiramente informativas) e ruído em 5. 

## 4. Separação entre Treino e Teste

In [26]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEMENTE
)

In [27]:
print(X_train.shape)

(4000, 100)


## 5. Implementando PCA

O PCA (em português, análise das componentes principais) é um método de redução de dimensionalidade baseado em transformações lineares muito utilizado em uma garnde diversidade de problemas científicos [4]. Por ser baseado em transformação linear, sua implementação é simples e seu custo computacional é baixo comparado ao uso de autoencoders para o mesmo propósito. 

In [28]:
pca = PCA(n_components=10)

X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

## 6. Implementando Autoencoder 

Conversão dos dados para tensores (necessário para uso de PyTorch)

In [29]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)

Conversão dos tensores para Datasets

In [30]:
train_dataset = TensorDataset(X_train_tensor)
test_dataset = TensorDataset(X_test_tensor)

E aqui convertemos os datasets para DataLoaders. Isso é útil por que o PyTorch usa esses DataLoaders pra treinar os modelos agrupando os dados em _batches_, o que permite paralelização do treino e maior eficiência [6].

In [31]:
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

Agora, criamos a instância da Rede Neural autoencoder, utilizando a classe mãe `nn.Module` do PyTorch. Para que o modelo treine, são necessários os encoder e decoder, mas apenas o encoder é aplicado para a redução de dimensionalidade. O decoder é necessário na etapa do treino para que o modelo possa comparar a reconstrução com o dado original, e assim calcular a perda e ajustar os parâmetros. [7]

In [32]:
class Autoencoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(100, 75),
            nn.ReLU(), 
            nn.Linear(75, 50), 
            nn.ReLU(), 
            nn.Linear(50, 25), 
            nn.ReLU(), 
            nn.Linear(25, 10), 
        )

        self.decoder = nn.Sequential(
            nn.Linear(10, 25),
            nn.ReLU(), 
            nn.Linear(25, 50), 
            nn.ReLU(), 
            nn.Linear(50, 75), 
            nn.ReLU(), 
            nn.Linear(75, 100), 
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return encoded, decoded


Nossa rede tem na primeira camada do encoder 100 neurônios, um para cada feature, e chega no _bottleneck_ com 10 features (aqui não há uma relação com o número de variáveis informativas). Essa redução no número de neurônios se dá de forma progressiva. Então, o decoder tem uma arquitetura inversa com relação ao encoder, e faz a reconstrução dos dados no número de dimensões oringinal. 

### 6.1 Configurações do Autoencoder

In [33]:
model = Autoencoder()
criterion = nn.MSELoss()
optimizer  = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay = 1e-5)

O weight decay faz com que os pesos sejam mantidos em valores pequenos, ajudando a evitar overfitting [6]. 

## 7. Loop de treino do Autoencoder

In [34]:
num_epochs = 100 # número escolhido de forma arbitrária

for epoch in range(num_epochs):

    for (features, ) in train_loader:

        enc, dec = model(features) # separo resultados de encoder e decoder

        loss = criterion(dec, features)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f'Epoch:{epoch+1}, Loss:{loss.item():.4f}')

with torch.no_grad(): # aqui eu configuro pra que os dados de treino e teste não passem pelo decoder quando os 
    # os modelos forem treinar e fazer previsões

    X_train_encoded = model.encoder(X_train_tensor)

    X_test_encoded = model.encoder(X_test_tensor)

Epoch:1, Loss:0.9652
Epoch:2, Loss:0.9571
Epoch:3, Loss:0.9986
Epoch:4, Loss:0.9075
Epoch:5, Loss:0.8986
Epoch:6, Loss:0.9346
Epoch:7, Loss:0.9099
Epoch:8, Loss:0.9141
Epoch:9, Loss:0.9091
Epoch:10, Loss:0.9177
Epoch:11, Loss:0.9620
Epoch:12, Loss:0.8973
Epoch:13, Loss:0.8881
Epoch:14, Loss:0.9163
Epoch:15, Loss:0.9043
Epoch:16, Loss:0.9033
Epoch:17, Loss:0.9073
Epoch:18, Loss:0.8896
Epoch:19, Loss:0.8826
Epoch:20, Loss:0.8971
Epoch:21, Loss:0.9146
Epoch:22, Loss:0.8994
Epoch:23, Loss:0.8968
Epoch:24, Loss:0.9161
Epoch:25, Loss:0.9066
Epoch:26, Loss:0.9159
Epoch:27, Loss:0.9369
Epoch:28, Loss:0.8913
Epoch:29, Loss:0.8994
Epoch:30, Loss:0.8583
Epoch:31, Loss:0.9071
Epoch:32, Loss:0.8987
Epoch:33, Loss:0.8611
Epoch:34, Loss:0.8829
Epoch:35, Loss:0.9026
Epoch:36, Loss:0.8471
Epoch:37, Loss:0.8854
Epoch:38, Loss:0.8906
Epoch:39, Loss:0.8546
Epoch:40, Loss:0.8615
Epoch:41, Loss:0.8603
Epoch:42, Loss:0.8553
Epoch:43, Loss:0.8857
Epoch:44, Loss:0.8991
Epoch:45, Loss:0.8639
Epoch:46, Loss:0.84

O número de 100 épocas é arbitrário. É importante configurar pra que quando os modelos forem receber os dados para treinar, eles recebam dados que não passaram pelo decoder (se os dados passarem pelo decoder, não estaríamos reduzindo a dimensionalidade). 

E aqui, finalmente, fazemos que os dados resultantes do encoder voltem para o formato de `Array Numpy`, pois é assim que os modelos devem receber os dados. 

In [35]:
X_train_encoded = X_train_encoded.numpy()
X_test_encoded = X_test_encoded.numpy()

## 8. Instanciado e fazendo previsões com os modelos de ML

Os hiperparâmetros dos modelos não foram configurados nesse estudo. Seus valores padrão podem ser encontrados em [8]. 

### 8.1  Floresta Aleatória com PCA

In [36]:
model = RandomForestRegressor()

In [37]:
model.fit(X_train_pca, y_train)

y_pred_PCA = model.predict(X_test_pca)

RMSE = root_mean_squared_error(y_test, y_pred_PCA)

print(f"O RMSE do modelo árvore de decisão foi de {RMSE} unidades de y.")

O RMSE do modelo árvore de decisão foi de 408.12153017773244 unidades de y.


### 8.2 Floresta Aleatória com Autoencoder

In [38]:
model.fit(X_train_encoded, y_train)

y_pred_AE = model.predict(X_test_encoded)

RMSE = root_mean_squared_error(y_test, y_pred_AE)

print(f"O RMSE do modelo árvore de decisão foi de {RMSE} unidades de y.")

O RMSE do modelo árvore de decisão foi de 395.94825113900555 unidades de y.


### 8.3 SVR com PCA

In [39]:
model_svr = SVR(
    kernel='rbf'
)

o `kernel = rbf` que significa "Radial Basis Function" é usado para medir a similaridade de pontos de forma não-linear, levando eles para um espaço de dimensão muito alta, em que eles podem ser separados linearmente [6,8]. Isso pode enriquecer o estudo, já que o PCA é linear, mas o Autoencoder não. 

In [44]:
model_svr.fit(X_train_pca, y_train)

y_pred_svr = model_svr.predict(X_test_pca)

RMSE = root_mean_squared_error(y_test, y_pred_svr)

print(f"O RMSE do modelo SVR foi de {RMSE} unidades de y.")

O RMSE do modelo SVR foi de 406.6402002819936 unidades de y.


### 8.4 SVR com Autoencoder

In [45]:
model_svr.fit(X_train_encoded, y_train)

y_pred_svr = model_svr.predict(X_test_encoded)

RMSE = root_mean_squared_error(y_test, y_pred_svr)

print(f"O RMSE do modelo SVR foi de {RMSE} unidades de y.")

O RMSE do modelo SVR foi de 401.9363508797934 unidades de y.


### 8.5 Regressão Linear com PCA

In [42]:
model_LR = LinearRegression()

model_LR.fit(X_train_pca, y_train)

y_pred_LR = model_LR.predict(X_test_pca)

RMSE = root_mean_squared_error(y_test, y_pred_LR)

print(f"O RMSE do modelo LR foi de {RMSE} unidades de y.")

O RMSE do modelo LR foi de 398.73917074508967 unidades de y.


### 8.6 Regressão Linear com Autoencoder

In [43]:
model_LR.fit(X_train_encoded, y_train)

y_pred_LR = model_LR.predict(X_test_encoded)

RMSE = root_mean_squared_error(y_test, y_pred_LR)

print(f"O RMSE do modelo LR foi de {RMSE} unidades de y.")

O RMSE do modelo LR foi de 390.7224872724399 unidades de y.


## 9. Discussão dos Resultados

Podemos observar que nesse caso, os modelos que foram treinados a partir dos dados de Autoencoders obtiveram um valor de RMSE um pouco menor do que os aqueles que usaram os dados advindos do PCA. Esse é um resultado inicial interessante, mas pode ter sido um caso particular de `noise = 20`, dado que em simulações com outros valores de ruídos e features informativas isso não aconteceu. Isso será melhor estudado a partir do arquivo `Autoencoders_PCA_estudo.py`. 

É interessante notar que esse resultado não é o mesmo obtido pela referência [3], que obteve desempenhos parecidos para modelos que usaram dados de métodos de redução de dimensionalidade diferentes. Essa referência argumenta que dado esse resultado, o PCA pode ser um método mais interessante, já que sua implementação é mais simples e seu custo computacional em geral é menor. 

Mas no caso de nosso estudo, pode ser que pelo fato de que os autoencoders capturam relações não lineares entre as variáveis, isso tenha beneficiado os modelos, e portanto, gerado resultados de RMSE menores. 

## 10. Referências

[1] CASSAR, Daniel. ATP-203 4.0 - Árvore de decisão e floresta aleatória.ipynb. 2025. Material didático disponibilizado no Teams no curso de Machine Learning, Ilum Escola de Ciência, Campinas.
 . Acesso em: 09 mai. 2026.

[2] MICHELUCCI, Umberto. An introduction to autoencoders. 2022. Disponível em: <https://arxiv.org/abs/2201.03898>. Acesso em: 20 abr. 2026

[3] FOURNIER, Quentin; ALOISE, Daniel. Empirical comparison between autoencoders and
traditional dimensionality reduction methods. 2021. Disponível em: <https://ieeexplore.ieee.org/abstract/document/8791727>. Acesso em 10. mai. 2026. 

[4] CASSAR, Daniel. ATP-203 8.0 - Redução de Dimensionalidade com PCA.ipynb. 2025. Material didático disponibilizado no Teams no curso de Machine Learning, Ilum Escola de Ciência, Campinas.
 . Acesso em: 10 mai. 2026.

[5] IBM. What is dimensionality reduction? IBM, s.d. Disponível em: https://www.ibm.com/topics/dimensionality-reduction
. Acesso em: 10 maio 2026.

[6] OPENAI. ChatGPT. 2026. Disponível em: https://chat.openai.com/. Acesso em: 10 mai. 2026.   

[7] Deep Learning
GOODFELLOW, Ian; BENGIO, Yoshua; COURVILLE, Aaron. Deep learning. Cambridge: MIT Press, 2016. Disponível em: http://www.deeplearningbook.org
. Acesso em: 20 abr. 2026.

[8] scikit-learn. Supervised learning. [S. l.], s.d. Disponível em: <https://scikit-learn.org/stable/supervised_learning.html>. Acesso em: 10 maio 2026.